# Streams


We start, as usual, by loading our ice magic.


In [ ]:
%load_ext ice.magic

## Motivation

CUDA operations - kernel launches, memory transfers, and synchronization calls - are, unless otherwise specified, submitted to a single ordered queue.
Operations in this queue execute one after another in submission order.
This simplifies reasoning about correctness, but means that the GPU can be left idle in certain situations instead of doing useful work.

Two common scenarios motivate departing from this default:

* **Multiple small kernels**: A kernel's grid may be too small to fully utilize the GPU's capabilities (around 10k threads for compute and 100k threads for main memory saturation).
  In this case, concurrent execution of multiple independent kernels can increase the GPU utilization.
* **Costly host–device transfers**: Transferring data between CPU and GPU - usually via PCIe - can take considerable time.
  Concurrently executing kernels (that don't rely on the data transferred) allows for overlapping transfers and computation, thus hiding the transfer cost.

CUDA **streams** enable both patterns.
Each stream is an ordered queue of GPU operations mostly following the concepts covered before:
Operations submitted to the same stream execute in order and can not overlap.

## Running Example

The code below is adapted from the FMA example in the [GPU architecture notebook](./04-gpu-architecture.ipynb).
However, instead of a single work package, four independent datasets are created and each is processed by a separate kernel performing slightly different computations.
Each kernel is launched with 21 blocks of 1024 threads each which means that only a quarter of the 84 streaming multiprocessors on an A40 can be used.
This is intentional: we want to examine the effect of kernels that are unable to fully occupy a single GPU.

The timing now covers host-to-device transfers, kernel execution, and device-to-host transfers, so that later improvements from overlap are reflected in the measured runtime.

In [ ]:
%%cuda -c code/streams/fma-baseline.cu -p -o profiles/streams-fma-baseline

__global__ void fmaKernel(float* data, float scale, float add, int numFMA, int numElements) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < numElements; i += blockDim.x * gridDim.x) {
        float acc = data[i];
        for (int r = 0; r < numFMA; ++r)
            acc = scale * acc + add;
        data[i] = acc;
    }
}

int main() {
    const int numBlocks = 21;
    const int numThreadsPerBlock = 1024;
    const int numElements = 128 * numBlocks * numThreadsPerBlock;

    float *dataA, *dataB, *dataC, *dataD;
    float *d_dataA, *d_dataB, *d_dataC, *d_dataD;

    checkCudaError(cudaMallocHost(&dataA, numElements * sizeof(float)));
    checkCudaError(cudaMallocHost(&dataB, numElements * sizeof(float)));
    checkCudaError(cudaMallocHost(&dataC, numElements * sizeof(float)));
    checkCudaError(cudaMallocHost(&dataD, numElements * sizeof(float)));

    checkCudaError(cudaMalloc(&d_dataA, numElements * sizeof(float)));
    checkCudaError(cudaMalloc(&d_dataB, numElements * sizeof(float)));
    checkCudaError(cudaMalloc(&d_dataC, numElements * sizeof(float)));
    checkCudaError(cudaMalloc(&d_dataD, numElements * sizeof(float)));

    for (int i = 0; i < numElements; ++i)
        dataA[i] = dataB[i] = dataC[i] = dataD[i] = (float)i;

    //# run an empty kernel once to mitigate startup effects
    fmaKernel<<<1, 1>>>(0, 0, 0, 0, 0);

    auto start = std::chrono::steady_clock::now();

    checkCudaError(cudaMemcpy(d_dataA, dataA, numElements * sizeof(float), cudaMemcpyHostToDevice));
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataA, 0.11f, 1.1f, 1100, numElements);
    checkCudaError(cudaMemcpy(dataA, d_dataA, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    checkCudaError(cudaMemcpy(d_dataB, dataB, numElements * sizeof(float), cudaMemcpyHostToDevice));
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataB, 0.12f, 1.2f, 1200, numElements);
    checkCudaError(cudaMemcpy(dataB, d_dataB, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    checkCudaError(cudaMemcpy(d_dataC, dataC, numElements * sizeof(float), cudaMemcpyHostToDevice));
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataC, 0.13f, 1.3f, 1300, numElements);
    checkCudaError(cudaMemcpy(dataC, d_dataC, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    checkCudaError(cudaMemcpy(d_dataD, dataD, numElements * sizeof(float), cudaMemcpyHostToDevice));
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataD, 0.14f, 1.4f, 1400, numElements);
    checkCudaError(cudaMemcpy(dataD, d_dataD, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    const std::chrono::duration<double> elapsed = std::chrono::steady_clock::now() - start;
    std::cout << "Time elapsed: " << elapsed.count() * 1e3 << " ms\n";

    checkCudaError(cudaFreeHost(dataA));
    checkCudaError(cudaFreeHost(dataB));
    checkCudaError(cudaFreeHost(dataC));
    checkCudaError(cudaFreeHost(dataD));

    checkCudaError(cudaFree(d_dataA));
    checkCudaError(cudaFree(d_dataB));
    checkCudaError(cudaFree(d_dataC));
    checkCudaError(cudaFree(d_dataD));
}

The [Nsight Systems profile](./profiles/streams-fma-baseline.nsys-rep) shows that all operations are **serialized**: each transfer and kernel execution completes before the next begins.
This is the baseline we will improve in the exercises below.

As the code is quite verbose, we first reduce its length using modern C++ features.

In [ ]:
%%cuda -c code/streams/fma-baseline-compact.cu

__global__ void fmaKernel(float* data, float scale, float add, int numFMA, int numElements) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < numElements; i += blockDim.x * gridDim.x) {
        float acc = data[i];
        for (int r = 0; r < numFMA; ++r)
            acc = scale * acc + add;
        data[i] = acc;
    }
}

int main() {
    const int numBlocks = 21;
    const int numThreadsPerBlock = 1024;
    const int numElements = 128 * numBlocks * numThreadsPerBlock;

    float *dataA, *dataB, *dataC, *dataD;
    float *d_dataA, *d_dataB, *d_dataC, *d_dataD;

    for (auto ptrOfPtr : {&dataA, &dataB, &dataC, &dataD})
        checkCudaError(cudaMallocHost(ptrOfPtr, numElements * sizeof(float)));

    for (auto ptrOfPtr : {&d_dataA, &d_dataB, &d_dataC, &d_dataD})
        checkCudaError(cudaMalloc(ptrOfPtr, numElements * sizeof(float)));

    for (int i = 0; i < numElements; ++i)
        dataA[i] = dataB[i] = dataC[i] = dataD[i] = (float)i;

    //# run an empty kernel once to mitigate startup effects
    fmaKernel<<<1, 1>>>(0, 0, 0, 0, 0);

    auto start = std::chrono::steady_clock::now();

    checkCudaError(cudaMemcpy(d_dataA, dataA, numElements * sizeof(float), cudaMemcpyHostToDevice));
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataA, 0.11f, 1.1f, 1100, numElements);
    checkCudaError(cudaMemcpy(dataA, d_dataA, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    checkCudaError(cudaMemcpy(d_dataB, dataB, numElements * sizeof(float), cudaMemcpyHostToDevice));
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataB, 0.12f, 1.2f, 1200, numElements);
    checkCudaError(cudaMemcpy(dataB, d_dataB, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    checkCudaError(cudaMemcpy(d_dataC, dataC, numElements * sizeof(float), cudaMemcpyHostToDevice));
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataC, 0.13f, 1.3f, 1300, numElements);
    checkCudaError(cudaMemcpy(dataC, d_dataC, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    checkCudaError(cudaMemcpy(d_dataD, dataD, numElements * sizeof(float), cudaMemcpyHostToDevice));
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataD, 0.14f, 1.4f, 1400, numElements);
    checkCudaError(cudaMemcpy(dataD, d_dataD, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    const std::chrono::duration<double> elapsed = std::chrono::steady_clock::now() - start;
    std::cout << "Time elapsed: " << elapsed.count() * 1e3 << " ms\n";

    for (auto ptr : {dataA, dataB, dataC, dataD})
        checkCudaError(cudaFreeHost(ptr));

    for (auto ptr : {d_dataA, d_dataB, d_dataC, d_dataD})
        checkCudaError(cudaFree(ptr));
}

## CUDA Streams

A CUDA stream is an ordered queue of GPU operations.
Operations enqueued into a stream are asynchronous memory operations (`cudaMemcpyAsync`, `cudaMemPrefetchAsync`, ...), kernel launches, and others.

Operations inside one stream are executed
  * **in order** of their submission, and
  * **without overlap** with other operations of the **same stream**.

Operations between different streams **may** overlap.

## Default Stream

Let's look at an example.

```c++
cudaMemcpy(..., cudaMemcpyHostToDevice);
kernelA<<<grid, block>>>();
kernelB<<<grid, block>>>();
kernelC<<<grid, block>>>();
cudaMemcpy(..., cudaMemcpyDeviceToHost);
```

Without specification, the implicitly targeted stream is the default stream, or stream 0.
Explicitly targeting this stream is also possible and yields equivalent behavior.

A kernel can be submitted to a specific stream by passing that stream as **4th argument of the execution configuration**.
The third argument, which we simply set to zero for this tutorial, can be used to specify the size of dynamically allocated shared memory per block.

```c++
cudaMemcpy(..., cudaMemcpyHostToDevice);
kernelA<<<grid, block, 0, /* stream = */ 0>>>();
kernelB<<<grid, block, 0, /* stream = */ 0>>>();
kernelC<<<grid, block, 0, /* stream = */ 0>>>();
cudaMemcpy(..., cudaMemcpyDeviceToHost);
```

The figure below shows a possible execution, similar to what Nsight Systems might show in its timeline visualization.
Note the ordering and the absence of overlap.

<div style="text-align: center;">
  <img src="./img/streams-default.png" width="960" style="background-color:white" alt="Default Stream Visualization">
</div>

## Non-Default Streams

Let's enqueue kernels B and C into separate streams.

```c++
cudaMemcpy(..., cudaMemcpyHostToDevice);
kernelA<<<grid, block>>>();
kernelB<<<grid, block, 0, stream_1>>>();
kernelC<<<grid, block, 0, stream_2>>>();
cudaMemcpy(..., cudaMemcpyDeviceToHost);
```

The figure below shows a possible execution.
Note the ordering within streams, the blocking nature of the default stream and the overlap between non-default streams:
The default stream enforces a global barrier: it waits for all non-default-stream operations to complete before executing, and non-default streams wait for the default stream to complete before proceeding.

<div style="text-align: center;">
  <img src="./img/streams-nondefault.png" width="960" style="background-color:white" alt="Nondefault Stream Visualization">
</div>

Moving all kernels to non-default streams:

```c++
cudaMemcpy(..., cudaMemcpyHostToDevice);
kernelA<<<grid, block, 0, stream_1>>>();
kernelB<<<grid, block, 0, stream_2>>>();
kernelC<<<grid, block, 0, stream_3>>>();
cudaMemcpy(..., cudaMemcpyDeviceToHost);
```

The figure below shows a possible execution.
Note the arbitrary execution order between streams.

<div style="text-align: center;">
  <img src="./img/streams-nondefault-2.png" width="960" style="background-color:white" alt="Nondefault Stream Visualization">
</div>

## Creating and Destroying Streams

The next step is learning how to manage streams.

`cudaStream_t` is the handle type for a CUDA stream.
Streams are created with `cudaStreamCreate` and destroyed with `cudaStreamDestroy`:

```c++
cudaStream_t stream;
checkCudaError(cudaStreamCreate(&stream));

// ... submit work to stream ...

checkCudaError(cudaStreamDestroy(stream));
```

Note that `cudaStreamCreate` takes a **pointer** to a stream handle; `cudaStreamDestroy` takes the handle **by value**.

A simple way to observe stream behavior is to print from kernels assigned to different streams.
Since kernels in different streams may execute concurrently, the output order is non-deterministic (but will be ordered in most cases for this simple example).

In [ ]:
%%cuda -c code/streams/hello-world.cu -p -o profiles/streams-hello-world

__global__ void printStream(int streamId) {
    printf("Hello world from stream %d\n", streamId);
}

int main() {
    const int numStreams = 5;

    for (int s = 0; s < numStreams; ++s) {
        cudaStream_t stream;

        checkCudaError(cudaStreamCreate(&stream));
        printStream<<<1, 1, 0, stream>>>(s);
        checkCudaError(cudaStreamDestroy(stream));
    }

    cudaDeviceSynchronize();
}

Profiling the above example with Nsight Systems yields a report file ([profiles/streams-hello-world.nsys-rep](./profiles/streams-hello-world.nsys-rep)) that can be downloaded and visualized locally.
Note the additional stream rows in the GPU section.

## Stream Synchronization

`cudaDeviceSynchronize` will synchronize **all** streams.
In cases where more fine-grained synchronization is required, `cudaStreamSynchronize` can be used.
Check the below example - the streams are now handled as an array of handles since all stream handles need to be available in the synchronization loop.

In [ ]:
%%cuda -c code/streams/hello-world-sync.cu

__global__ void printStream(int streamId) {
    printf("Hello world from stream %d\n", streamId);
}

int main() {
    const int numStreams = 5;
    cudaStream_t streams[numStreams];

    for (int s = 0; s < numStreams; ++s)
        checkCudaError(cudaStreamCreate(&streams[s]));

    for (int s = 0; s < numStreams; ++s)
        printStream<<<1, 1, 0, streams[s]>>>(s);

    for (int s = 0; s < numStreams; ++s)
        checkCudaError(cudaStreamSynchronize(streams[s]));

    for (int s = 0; s < numStreams; ++s)
        checkCudaError(cudaStreamDestroy(streams[s]));

    cudaDeviceSynchronize();
}

## Exercise: Concurrent Kernel Execution

Let's return to our FMA example.
Each of the four dataset computations is independent: no kernel reads data written by another.
This makes them good candidates for concurrent execution.

Your tasks are:
* Create four CUDA streams.
* Launch each kernel in its own stream.
* Keep the memory transfers on the default stream for now - they will be addressed in the next exercise.
* Destroy the streams at the end.
* Download the report file ([profiles/streams-ex-kernels.nsys-rep](./profiles/streams-ex-kernels.nsys-rep)) and check for overlap.

If you follow these instructions closely, there is a good chance that you don't see any overlap.
Think for a moment why that is, how this relates to the mechanics of the default stream, and how to fix this.
Then:
* Apply your fix (hint: reorder the operations to have three distinct groups).
* Download the updated report file (still [profiles/streams-ex-kernels.nsys-rep](./profiles/streams-ex-kernels.nsys-rep)) and check for overlap.
* Compare the measured runtime to the baseline.

In [ ]:
%%cuda -c code/streams/ex-kernels.cu -p -o profiles/streams-ex-kernels

__global__ void fmaKernel(float* data, float scale, float add, int numFMA, int numElements) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < numElements; i += blockDim.x * gridDim.x) {
        float acc = data[i];
        for (int r = 0; r < numFMA; ++r)
            acc = scale * acc + add;
        data[i] = acc;
    }
}

int main() {
    const int numBlocks = 21;
    const int numThreadsPerBlock = 1024;
    const int numElements = 128 * numBlocks * numThreadsPerBlock;

    float *dataA, *dataB, *dataC, *dataD;
    float *d_dataA, *d_dataB, *d_dataC, *d_dataD;

    for (auto p : {&dataA, &dataB, &dataC, &dataD})
        checkCudaError(cudaMallocHost(p, numElements * sizeof(float)));

    for (auto p : {&d_dataA, &d_dataB, &d_dataC, &d_dataD})
        checkCudaError(cudaMalloc(p, numElements * sizeof(float)));

    for (int i = 0; i < numElements; ++i)
        dataA[i] = dataB[i] = dataC[i] = dataD[i] = (float)i;

    //# TODO: create four streams (streamA, streamB, streamC, streamD)

    //# run an empty kernel once to mitigate startup effects
    fmaKernel<<<1, 1>>>(0, 0, 0, 0, 0);

    auto start = std::chrono::steady_clock::now();

    checkCudaError(cudaMemcpy(d_dataA, dataA, numElements * sizeof(float), cudaMemcpyHostToDevice));
    //# TODO: launch fmaKernel for dataA in streamA instead of the default stream
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataA, 0.11f, 1.1f, 1100, numElements);
    checkCudaError(cudaMemcpy(dataA, d_dataA, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    checkCudaError(cudaMemcpy(d_dataB, dataB, numElements * sizeof(float), cudaMemcpyHostToDevice));
    //# TODO: launch fmaKernel for dataB in streamB instead of the default stream
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataB, 0.12f, 1.2f, 1200, numElements);
    checkCudaError(cudaMemcpy(dataB, d_dataB, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    checkCudaError(cudaMemcpy(d_dataC, dataC, numElements * sizeof(float), cudaMemcpyHostToDevice));
    //# TODO: launch fmaKernel for dataC in streamC instead of the default stream
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataC, 0.13f, 1.3f, 1300, numElements);
    checkCudaError(cudaMemcpy(dataC, d_dataC, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    checkCudaError(cudaMemcpy(d_dataD, dataD, numElements * sizeof(float), cudaMemcpyHostToDevice));
    //# TODO: launch fmaKernel for dataD in streamD instead of the default stream
    fmaKernel<<<numBlocks, numThreadsPerBlock>>>(d_dataD, 0.14f, 1.4f, 1400, numElements);
    checkCudaError(cudaMemcpy(dataD, d_dataD, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    const std::chrono::duration<double> elapsed = std::chrono::steady_clock::now() - start;
    std::cout << "Time elapsed: " << elapsed.count() * 1e3 << " ms\n";

    for (auto p : {dataA, dataB, dataC, dataD})
        checkCudaError(cudaFreeHost(p));

    for (auto p : {d_dataA, d_dataB, d_dataC, d_dataD})
        checkCudaError(cudaFree(p));

    //# TODO: destroy streams
}

### Possible Solution

In [ ]:
%%cuda -c code/streams/sol-kernels.cu -p -o profiles/streams-sol-kernels

__global__ void fmaKernel(float* data, float scale, float add, int numFMA, int numElements) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < numElements; i += blockDim.x * gridDim.x) {
        float acc = data[i];
        for (int r = 0; r < numFMA; ++r)
            acc = scale * acc + add;
        data[i] = acc;
    }
}

int main() {
    const int numBlocks = 21;
    const int numThreadsPerBlock = 1024;
    const int numElements = 128 * numBlocks * numThreadsPerBlock;

    float *dataA, *dataB, *dataC, *dataD;
    float *d_dataA, *d_dataB, *d_dataC, *d_dataD;

    for (auto p : {&dataA, &dataB, &dataC, &dataD})
        checkCudaError(cudaMallocHost(p, numElements * sizeof(float)));

    for (auto p : {&d_dataA, &d_dataB, &d_dataC, &d_dataD})
        checkCudaError(cudaMalloc(p, numElements * sizeof(float)));

    for (int i = 0; i < numElements; ++i)
        dataA[i] = dataB[i] = dataC[i] = dataD[i] = (float)i;

    cudaStream_t streamA, streamB, streamC, streamD;
    checkCudaError(cudaStreamCreate(&streamA));
    checkCudaError(cudaStreamCreate(&streamB));
    checkCudaError(cudaStreamCreate(&streamC));
    checkCudaError(cudaStreamCreate(&streamD));

    //# run an empty kernel once to mitigate startup effects
    fmaKernel<<<1, 1>>>(0, 0, 0, 0, 0);

    auto start = std::chrono::steady_clock::now();

    //# H2D transfers - default stream
    checkCudaError(cudaMemcpy(d_dataA, dataA, numElements * sizeof(float), cudaMemcpyHostToDevice));
    checkCudaError(cudaMemcpy(d_dataB, dataB, numElements * sizeof(float), cudaMemcpyHostToDevice));
    checkCudaError(cudaMemcpy(d_dataC, dataC, numElements * sizeof(float), cudaMemcpyHostToDevice));
    checkCudaError(cudaMemcpy(d_dataD, dataD, numElements * sizeof(float), cudaMemcpyHostToDevice));

    //# each kernel in its own stream
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamA>>>(d_dataA, 0.11f, 1.1f, 1100, numElements);
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamB>>>(d_dataB, 0.12f, 1.2f, 1200, numElements);
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamC>>>(d_dataC, 0.13f, 1.3f, 1300, numElements);
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamD>>>(d_dataD, 0.14f, 1.4f, 1400, numElements);

    //# wait for all kernels to complete before D2H transfers
    checkCudaError(cudaStreamSynchronize(streamA));
    checkCudaError(cudaStreamSynchronize(streamB));
    checkCudaError(cudaStreamSynchronize(streamC));
    checkCudaError(cudaStreamSynchronize(streamD));

    //# D2H transfers - default stream
    checkCudaError(cudaMemcpy(dataA, d_dataA, numElements * sizeof(float), cudaMemcpyDeviceToHost));
    checkCudaError(cudaMemcpy(dataB, d_dataB, numElements * sizeof(float), cudaMemcpyDeviceToHost));
    checkCudaError(cudaMemcpy(dataC, d_dataC, numElements * sizeof(float), cudaMemcpyDeviceToHost));
    checkCudaError(cudaMemcpy(dataD, d_dataD, numElements * sizeof(float), cudaMemcpyDeviceToHost));

    const std::chrono::duration<double> elapsed = std::chrono::steady_clock::now() - start;
    std::cout << "Time elapsed: " << elapsed.count() * 1e3 << " ms\n";

    checkCudaError(cudaStreamDestroy(streamA));
    checkCudaError(cudaStreamDestroy(streamB));
    checkCudaError(cudaStreamDestroy(streamC));
    checkCudaError(cudaStreamDestroy(streamD));

    for (auto p : {dataA, dataB, dataC, dataD})
        checkCudaError(cudaFreeHost(p));

    for (auto p : {d_dataA, d_dataB, d_dataC, d_dataD})
        checkCudaError(cudaFree(p));
}

Visualizing our application's time line after downloading the report file ([profiles/streams-sol-kernels.nsys-rep](./profiles/streams-sol-kernels.nsys-rep)) reveals concurrent execution of the four kernels.
Memory transfers are still ordered and non-overlapping as they are still executed in the default stream - this is our next optimization target.

## Streams for Data Transfers

The memory transfers in our example so far use `cudaMemcpy`, which **blocks the calling host thread** until the transfer is complete (for most cases: [documentation](https://docs.nvidia.com/cuda/cuda-runtime-api/api-sync-behavior.html#api-sync-behavior__memcpy-sync)).
It accepts no stream argument and always operates on the default stream, making it impossible to overlap transfers with other GPU work.

`cudaMemcpyAsync` is the stream-aware counterpart.
It enqueues the transfer as an operation in a given stream and **returns immediately**, and the CPU can continue submitting work while the transfer runs in the background.

```c++
cudaError_t cudaMemcpyAsync(
    void*          dst,
    const void*    src,
    size_t         count,
    cudaMemcpyKind kind,
    cudaStream_t   stream = 0
);
```

Note the similarity with `cudaMemcpy` - the key change is an **optional** fifth argument that defaults to the 0-stream (default stream) if it is not set.

For `cudaMemcpyAsync` to be able to actually overlap with other work, the host memory must be **page-locked** (pinned).
Transfers from ordinary heap memory (`malloc`, `new`) silently use staging via page-locked allocations which decreases performance and prevents overlap.
`cudaMallocHost` allocates page-locked host memory - this is why all arrays in the running example are already allocated with it.

For **managed memory** allocated with `cudaMallocManaged`, `cudaMemPrefetchAsync` serves the same purpose: it migrates a memory region to a target device asynchronously and also accepts a stream as an optional argument.


## Exercise: Overlapping Transfers

In the previous exercise, all memory transfers still ran on the default stream, so they remained serialized.
Switching to asynchronous transfers (`cudaMemcpyAsync`) and assigning them to the same stream as their corresponding kernel allows transfers and kernels across different work packages to overlap.

An important constraint: within each stream, operations execute in the order they are submitted.
The H2D transfer, kernel launch, and D2H transfer for the **same work package** must all be submitted to the **same stream**.
This guarantees the kernel waits for its input data, and the D2H copy runs only after the kernel has written its results.

Your tasks are:
* Think about which ordering of operations is more beneficial - the grouped one from the last exercise or the initial pipeline one?
* When in doubt, implement and profile both (if not, then do it anyway).
  Check the report file ([profiles/streams-ex-transfers.nsys-rep](./profiles/streams-ex-transfers.nsys-rep)).
  Do the results match your expectation?

In [ ]:
%%cuda -c code/streams/ex-transfers.cu -p -o profiles/streams-ex-transfers

__global__ void fmaKernel(float* data, float scale, float add, int numFMA, int numElements) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < numElements; i += blockDim.x * gridDim.x) {
        float acc = data[i];
        for (int r = 0; r < numFMA; ++r)
            acc = scale * acc + add;
        data[i] = acc;
    }
}

int main() {
    const int numBlocks = 21;
    const int numThreadsPerBlock = 1024;
    const int numElements = 128 * numBlocks * numThreadsPerBlock;

    float *dataA, *dataB, *dataC, *dataD;
    float *d_dataA, *d_dataB, *d_dataC, *d_dataD;

    for (auto p : {&dataA, &dataB, &dataC, &dataD})
        checkCudaError(cudaMallocHost(p, numElements * sizeof(float)));

    for (auto p : {&d_dataA, &d_dataB, &d_dataC, &d_dataD})
        checkCudaError(cudaMalloc(p, numElements * sizeof(float)));

    for (int i = 0; i < numElements; ++i)
        dataA[i] = dataB[i] = dataC[i] = dataD[i] = (float)i;

    cudaStream_t streamA, streamB, streamC, streamD;
    for (auto p : {&streamA, &streamB, &streamC, &streamD})
        checkCudaError(cudaStreamCreate(p));

    //# run an empty kernel once to mitigate startup effects
    fmaKernel<<<1, 1>>>(0, 0, 0, 0, 0);

    auto start = std::chrono::steady_clock::now();

    //# TODO: for each work package (A/B/C/D), submit H2D, kernel, and D2H to the same stream.
    //#       Replace cudaMemcpy with cudaMemcpyAsync.

    for (auto p : {streamA, streamB, streamC, streamD})
        checkCudaError(cudaStreamSynchronize(p));

    const std::chrono::duration<double> elapsed = std::chrono::steady_clock::now() - start;
    std::cout << "Time elapsed: " << elapsed.count() * 1e3 << " ms\n";

    for (auto p : {dataA, dataB, dataC, dataD})
        checkCudaError(cudaFreeHost(p));

    for (auto p : {d_dataA, d_dataB, d_dataC, d_dataD})
        checkCudaError(cudaFree(p));

    for (auto p : {streamA, streamB, streamC, streamD})
        checkCudaError(cudaStreamDestroy(p));
}

### Possible Solution


The **grouped** approach submits all H2D transfers first, then all kernel launches, then all D2H transfers - operations of the same type are grouped across all four streams.


In [ ]:
%%cuda -c code/streams/sol-transfers-grouped.cu -p -o profiles/streams-sol-transfers-grouped

__global__ void fmaKernel(float* data, float scale, float add, int numFMA, int numElements) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < numElements; i += blockDim.x * gridDim.x) {
        float acc = data[i];
        for (int r = 0; r < numFMA; ++r)
            acc = scale * acc + add;
        data[i] = acc;
    }
}

int main() {
    const int numBlocks = 21;
    const int numThreadsPerBlock = 1024;
    const int numElements = 128 * numBlocks * numThreadsPerBlock;

    float *dataA, *dataB, *dataC, *dataD;
    float *d_dataA, *d_dataB, *d_dataC, *d_dataD;

    for (auto p : {&dataA, &dataB, &dataC, &dataD})
        checkCudaError(cudaMallocHost(p, numElements * sizeof(float)));

    for (auto p : {&d_dataA, &d_dataB, &d_dataC, &d_dataD})
        checkCudaError(cudaMalloc(p, numElements * sizeof(float)));

    for (int i = 0; i < numElements; ++i)
        dataA[i] = dataB[i] = dataC[i] = dataD[i] = (float)i;

    cudaStream_t streamA, streamB, streamC, streamD;
    for (auto p : {&streamA, &streamB, &streamC, &streamD})
        checkCudaError(cudaStreamCreate(p));

    //# run an empty kernel once to mitigate startup effects
    fmaKernel<<<1, 1>>>(0, 0, 0, 0, 0);

    auto start = std::chrono::steady_clock::now();

    //# H2D transfers - all work packages, each in its own stream
    checkCudaError(cudaMemcpyAsync(d_dataA, dataA, numElements * sizeof(float), cudaMemcpyHostToDevice, streamA));
    checkCudaError(cudaMemcpyAsync(d_dataB, dataB, numElements * sizeof(float), cudaMemcpyHostToDevice, streamB));
    checkCudaError(cudaMemcpyAsync(d_dataC, dataC, numElements * sizeof(float), cudaMemcpyHostToDevice, streamC));
    checkCudaError(cudaMemcpyAsync(d_dataD, dataD, numElements * sizeof(float), cudaMemcpyHostToDevice, streamD));

    //# kernels - each in its own stream (intra-stream ordering ensures H2D is done first)
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamA>>>(d_dataA, 0.11f, 1.1f, 1100, numElements);
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamB>>>(d_dataB, 0.12f, 1.2f, 1200, numElements);
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamC>>>(d_dataC, 0.13f, 1.3f, 1300, numElements);
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamD>>>(d_dataD, 0.14f, 1.4f, 1400, numElements);

    //# D2H transfers - each in its own stream (intra-stream ordering ensures kernel is done first)
    checkCudaError(cudaMemcpyAsync(dataA, d_dataA, numElements * sizeof(float), cudaMemcpyDeviceToHost, streamA));
    checkCudaError(cudaMemcpyAsync(dataB, d_dataB, numElements * sizeof(float), cudaMemcpyDeviceToHost, streamB));
    checkCudaError(cudaMemcpyAsync(dataC, d_dataC, numElements * sizeof(float), cudaMemcpyDeviceToHost, streamC));
    checkCudaError(cudaMemcpyAsync(dataD, d_dataD, numElements * sizeof(float), cudaMemcpyDeviceToHost, streamD));

    for (auto p : {streamA, streamB, streamC, streamD})
        checkCudaError(cudaStreamSynchronize(p));

    const std::chrono::duration<double> elapsed = std::chrono::steady_clock::now() - start;
    std::cout << "Time elapsed: " << elapsed.count() * 1e3 << " ms\n";

    for (auto p : {dataA, dataB, dataC, dataD})
        checkCudaError(cudaFreeHost(p));

    for (auto p : {d_dataA, d_dataB, d_dataC, d_dataD})
        checkCudaError(cudaFree(p));

    for (auto p : {streamA, streamB, streamC, streamD})
        checkCudaError(cudaStreamDestroy(p));
}

The **pipeline** approach interleaves the three operations per work package: for each dataset, H2D, kernel, and D2H are submitted in sequence to its stream before moving on to the next.


In [ ]:
%%cuda -c code/streams/sol-transfers-pipelines.cu -p -o profiles/streams-sol-transfers-pipelines

__global__ void fmaKernel(float* data, float scale, float add, int numFMA, int numElements) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < numElements; i += blockDim.x * gridDim.x) {
        float acc = data[i];
        for (int r = 0; r < numFMA; ++r)
            acc = scale * acc + add;
        data[i] = acc;
    }
}

int main() {
    const int numBlocks = 21;
    const int numThreadsPerBlock = 1024;
    const int numElements = 128 * numBlocks * numThreadsPerBlock;

    float *dataA, *dataB, *dataC, *dataD;
    float *d_dataA, *d_dataB, *d_dataC, *d_dataD;

    for (auto p : {&dataA, &dataB, &dataC, &dataD})
        checkCudaError(cudaMallocHost(p, numElements * sizeof(float)));

    for (auto p : {&d_dataA, &d_dataB, &d_dataC, &d_dataD})
        checkCudaError(cudaMalloc(p, numElements * sizeof(float)));

    for (int i = 0; i < numElements; ++i)
        dataA[i] = dataB[i] = dataC[i] = dataD[i] = (float)i;

    cudaStream_t streamA, streamB, streamC, streamD;
    for (auto p : {&streamA, &streamB, &streamC, &streamD})
        checkCudaError(cudaStreamCreate(p));

    //# run an empty kernel once to mitigate startup effects
    fmaKernel<<<1, 1>>>(0, 0, 0, 0, 0);

    auto start = std::chrono::steady_clock::now();

    //# H2D, kernel, D2H for each work package - each submitted to its own stream
    checkCudaError(cudaMemcpyAsync(d_dataA, dataA, numElements * sizeof(float), cudaMemcpyHostToDevice, streamA));
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamA>>>(d_dataA, 0.11f, 1.1f, 1100, numElements);
    checkCudaError(cudaMemcpyAsync(dataA, d_dataA, numElements * sizeof(float), cudaMemcpyDeviceToHost, streamA));

    checkCudaError(cudaMemcpyAsync(d_dataB, dataB, numElements * sizeof(float), cudaMemcpyHostToDevice, streamB));
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamB>>>(d_dataB, 0.12f, 1.2f, 1200, numElements);
    checkCudaError(cudaMemcpyAsync(dataB, d_dataB, numElements * sizeof(float), cudaMemcpyDeviceToHost, streamB));

    checkCudaError(cudaMemcpyAsync(d_dataC, dataC, numElements * sizeof(float), cudaMemcpyHostToDevice, streamC));
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamC>>>(d_dataC, 0.13f, 1.3f, 1300, numElements);
    checkCudaError(cudaMemcpyAsync(dataC, d_dataC, numElements * sizeof(float), cudaMemcpyDeviceToHost, streamC));

    checkCudaError(cudaMemcpyAsync(d_dataD, dataD, numElements * sizeof(float), cudaMemcpyHostToDevice, streamD));
    fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streamD>>>(d_dataD, 0.14f, 1.4f, 1400, numElements);
    checkCudaError(cudaMemcpyAsync(dataD, d_dataD, numElements * sizeof(float), cudaMemcpyDeviceToHost, streamD));

    for (auto p : {streamA, streamB, streamC, streamD})
        checkCudaError(cudaStreamSynchronize(p));

    const std::chrono::duration<double> elapsed = std::chrono::steady_clock::now() - start;
    std::cout << "Time elapsed: " << elapsed.count() * 1e3 << " ms\n";

    for (auto p : {dataA, dataB, dataC, dataD})
        checkCudaError(cudaFreeHost(p));

    for (auto p : {d_dataA, d_dataB, d_dataC, d_dataD})
        checkCudaError(cudaFree(p));

    for (auto p : {streamA, streamB, streamC, streamD})
        checkCudaError(cudaStreamDestroy(p));
}

The report files for the grouped approach ([profiles/streams-sol-transfers-grouped.nsys-rep](./profiles/streams-sol-transfers-grouped.nsys-rep)) and pipeline approach ([profiles/streams-sol-transfers-pipelines.nsys-rep](./profiles/streams-sol-transfers-pipelines.nsys-rep)) show a similar picture:
With all three operations for each work package in the same stream, the GPU can now overlap transfers from one stream with kernel execution in another.
Operations within each stream (H2D, kernel, D2H) are still ordered and non-overlapping.

## Outlook: Chunking a Single Dataset

The previous exercises operated on four *independent* datasets.
A different scenario is a single large dataset processed by a single kernel.
Is stream-based overlap still useful?

Yes, provided two conditions hold:
* The kernel runtime and transfer time are of **comparable magnitude**.
  If one dominates completely, there is little room to hide the other.
* The data can be split into independent **chunks** with no dependencies between them.
  The FMA kernel satisfies this: each element is processed in isolation.
  (in case of localized data dependencies, additional more complicated alternatives are available.)

The idea is to pipeline the work: while stream *n* runs its kernel, stream *n+1* can already be transferring its chunk to the device.

The code below demonstrates the pattern.
An array of stream handles makes it straightforward to vary the number of chunks without changing the loop logic.


In [ ]:
%%cuda -c code/streams/fma-chunked.cu -p -o profiles/streams-fma-chunked

__global__ void fmaKernel(float* data, float scale, float add, int numFMA, int numElements) {
    for (int i = blockIdx.x * blockDim.x + threadIdx.x; i < numElements; i += blockDim.x * gridDim.x) {
        float acc = data[i];
        for (int r = 0; r < numFMA; ++r)
            acc = scale * acc + add;
        data[i] = acc;
    }
}

int main() {
    const int numBlocks = 21;
    const int numThreadsPerBlock = 1024;
    const int numChunks = 16;
    const int totalElements = 128 * 4 * numBlocks * numThreadsPerBlock;  //# 4 matches the number of datasets in the running example
    const int chunkElements = totalElements / numChunks;
    const size_t chunkBytes = chunkElements * sizeof(float);

    float* data;
    checkCudaError(cudaMallocHost(&data, totalElements * sizeof(float)));

    float* d_data;
    checkCudaError(cudaMalloc(&d_data, totalElements * sizeof(float)));

    for (int i = 0; i < totalElements; ++i)
        data[i] = (float)i;

    //# heap-allocated stream array accommodates a variable numChunks
    cudaStream_t* streams = new cudaStream_t[numChunks];
    for (int i = 0; i < numChunks; ++i)
        checkCudaError(cudaStreamCreate(&streams[i]));

    //# run an empty kernel once to mitigate startup effects
    fmaKernel<<<1, 1>>>(0, 0, 0, 0, 0);

    auto start = std::chrono::steady_clock::now();

    for (int i = 0; i < numChunks; ++i) {
        const int offset = i * chunkElements;
        checkCudaError(cudaMemcpyAsync(d_data + offset, data + offset, chunkBytes, cudaMemcpyHostToDevice, streams[i]));
        fmaKernel<<<numBlocks, numThreadsPerBlock, 0, streams[i]>>>(d_data + offset, 0.12f, 1.2f, 1600, chunkElements);
        checkCudaError(cudaMemcpyAsync(data + offset, d_data + offset, chunkBytes, cudaMemcpyDeviceToHost, streams[i]));
    }

    for (int i = 0; i < numChunks; ++i)
        checkCudaError(cudaStreamSynchronize(streams[i]));

    const std::chrono::duration<double> elapsed = std::chrono::steady_clock::now() - start;
    std::cout << "Time elapsed: " << elapsed.count() * 1e3 << " ms\n";

    for (int i = 0; i < numChunks; ++i)
        checkCudaError(cudaStreamDestroy(streams[i]));
    delete[] streams;

    checkCudaError(cudaFreeHost(data));
    checkCudaError(cudaFree(d_data));
}

The report file ([profiles/streams-fma-chunked.nsys-rep](./profiles/streams-fma-chunked.nsys-rep)) shows behavior similar to the other cases of this notebook.

## Next Step

Head back to the [summary & outlook](./06-summary-outlook.ipynb) notebook.